In [1]:
# === SETUP: Run this first! ===
import os
import sys

# Change to project root and add to Python path
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)  # Goes up one level from 'notebooks/'
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"tsnn module path: {os.path.join(project_root, 'tsnn')}")

Project root: /Users/cyrilgarcia/notebooks/tsnn
tsnn module path: /Users/cyrilgarcia/notebooks/tsnn/tsnn


In [2]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV, LinearRegression
import torch
from torch.utils.data import Dataset, random_split
from torch.utils.data import DataLoader
import importlib
import sys
from tqdm import tqdm
#sys.path.append('/Users/cyrilgarcia/notebooks/tsnn/')

import tsnn

from tsnn.generators import generators
from tsnn.benchmarks import benchmark_comparison, ml_benchmarks, torch_benchmarks
from tsnn import utils
import torch.nn.functional as F
import math
from typing import Optional
from tsnn.tstorch import transformers
from sklearn.linear_model import LinearRegression, ElasticNetCV, ElasticNet




plt.style.use('ggplot')

In [3]:
from dataclasses import dataclass
from torch import nn


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")

Using device: mps


In [5]:
from typing import Dict

In [6]:
from tsnn.tstorch import models

In [7]:
from tsnn.tstorch.models import GlobalMLP, BiDimensionalMLP, OneDimensionalTransformer, CustomBiDimensionalTransformer
from sklearn.ensemble import HistGradientBoostingRegressor



# Summary

In [8]:
# In this notebook we will run the experiments to generate the figures for the paper.

# Test dataset

In [9]:
# We work with the following data.

In [10]:
# Global parameters
T_max = 5000
N1 = 10
F1 = 20
T1 = 5 # This parameter will be the n_rolling

print(T_max, N1, F1, T1)

5000 10 20 5


In [11]:
def generate_synthetic_datasets(
    num_time_steps: int = 3000,
    num_time_series: int = 10,
    num_features: int = 10,
    low_corr: float = 0.1,
    high_corr: float = 0.2,
    pct_zero_corr: float = 0.5,
) -> Dict[str, "generators.Generator"]:
    """
    Generates 5 synthetic multivariate time series datasets with different
    types of cross-series dependencies.

    Returns
    -------
    dict
        Keys: "d_lin", "d_cond", "d_shift", "d_cs", "d_cs_shift", "d_all"
        Values: generators.Generator objects (already with .train and .test)
    """
    dic_data = {}

    # Helper to avoid repeating the same 10 lines
    def make_gen(split_conditional=0.0,
                 split_shift=0.0,
                 split_seasonal=0.0,
                 split_cs=0.0,
                 split_cs_shift=0.0):
        gen = generators.Generator(num_time_steps, num_time_series, num_features)
        gen.generate_dataset(
            pct_zero_corr=pct_zero_corr,
            split_conditional=split_conditional,
            split_shift=split_shift,
            split_seasonal=split_seasonal,
            split_cs=split_cs,
            split_cs_shift=split_cs_shift,
            low_corr=low_corr,
            high_corr=high_corr,
        )
        return gen

    dic_data["d_lin"] = make_gen()

    # 1. Pure conditional (causal) dependence
    dic_data["d_cond"] = make_gen(split_conditional=1.0)

    # 2. Pure lagged (time-shifted) dependence
    dic_data["d_shift"] = make_gen(split_shift=1.0)

    # 3. Pure contemporaneous cross-sectional correlation
    dic_data["d_cs"] = make_gen(split_cs=1.0)

    # 4. Contemporaneous + lagged cross-series
    dic_data["d_cs_shift"] = make_gen(split_cs_shift=1.0)

    # 5. Equal mix of all four mechanisms
    dic_data["d_all"] = make_gen(
        split_conditional=0.2,
        split_shift=0.2,
        split_cs=0.2,
        split_cs_shift=0.2,
    )

    return dic_data

In [12]:
# list_low_corr = [0.01, 0.025, 0.05, 0.1]
# list_high_corr = [2*x for x in list_low_corr]

list_low_corr = [0.01, 0.03, 0.05, 0.1, 0.3]
list_high_corr = list_low_corr

dic_data = {}

for i in range(len(list_low_corr)):
    name = "correl" + str(list_low_corr[i])
    dic_data[name] = generate_synthetic_datasets(num_time_steps=T_max, num_time_series=N1, num_features=F1, low_corr=list_low_corr[i], high_corr=list_high_corr[i])
    

In [13]:
# We will fix the above dataset for now.

In [14]:
def causal_mask(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx

def build_attention_mask(mask_fn, seq_len, device="cpu"):
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device)
    h = torch.zeros(1, device=device)
    mask_bool = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])  # (seq_len, seq_len)
    return mask_bool

mask = causal_mask
mask = build_attention_mask(mask, T1, device=device)
def custom_mask_mod(b, h, q_idx, kv_idx):
    return mask[q_idx, kv_idx]

# List of models

In [15]:
# Let's list here all the models we wish to test on all the data.

In [16]:
def get_models():
    MLP_global = GlobalMLP(N1, F1, T1, dropout=0.2).to(device)

    MLP_2D = BiDimensionalMLP(N1, F1, T1, dropout=0.2).to(device)

    trans_1D_T4 = OneDimensionalTransformer(N1, F1, T1, mask=mask, attn_direction="T",  num_attn_layers=4,
                                            dropout=0.2, roll_y=True).to(device)
    #Note: using the MLP compression seems very bad..

    trans_2D_TCTC = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=True).to(device)

    trans_2D_TCTCTCTC = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTCTCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=True).to(device)


    dic_models = {'MLP_global':MLP_global, 'MLP_2D':MLP_2D, "trans_1D_T4":trans_1D_T4, "trans_2D_TCTC":trans_2D_TCTC, "trans_2D_TCTCTCTC":trans_2D_TCTCTCTC}

    trans_2D_TCTC_rollfalse = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=False).to(device)

    trans_2D_TCTCTCTC_rollfalse = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTCTCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=False).to(device)


    dic_models_rollfalse = {"trans_2D_TCTC":trans_2D_TCTC_rollfalse, "trans_2D_TCTCTCTC":trans_2D_TCTCTCTC_rollfalse}
    
    return dic_models, dic_models_rollfalse

In [17]:
# For each model we also need to specify the option we will use to fit them.

# Function to create table for an effect

In [18]:
# We give the function that creates for a given effect the table testing all models and all noise level.

In [19]:
def run_sim(model, n=200, N=N1*T1*F1, T=T_max, rho=0, oos=True):
    res = 0
    for k in tqdm(range(n)):
        
        # constant rho_i case
        rho_i = np.array([rho * np.sqrt(1 / N) for k in range(N)])

        # random rho_i re-normalized
        rho_i = np.random.uniform(low=-1, high=1, size=N)
        rho_i = np.random.normal(size=N)
        norm_factor = np.sum([rho_i[k]**2 for k in range(N)])
        rho_i = np.sqrt(np.abs(rho_i / norm_factor)) * np.sign(rho_i)
        rho_i = np.sqrt((rho_i**2) * (rho**2)) * np.sign(rho_i)


        mean = np.zeros(N)
        cov = np.eye(N)
        X = np.random.multivariate_normal(mean=mean, cov=cov, size=T)
        eps = np.random.normal(size=T)

        y_tilde = np.dot(X, rho_i)
        y = y_tilde + eps

        model.fit(X[:int(len(X)/2)], y[:int(len(X)/2)])
        if oos:
            res += np.corrcoef(model.predict(X[int(len(X)/2):]), y_tilde[int(len(X)/2):])[0][1]
        else:
            res += np.corrcoef(model.predict(X[:int(len(X)/2)]), y_tilde[:int(len(X)/2):])[0][1]
    return res / n

def get_ols_corr( N=N1*T1*F1, T=T_max, rho=0):
    return rho / np.sqrt(rho**2 + (1-rho**2) * min(N, T) / T)


def run_models(effect1):

    # Storage
    records_train = []
    records_test  = []

    for noise_level in list(dic_data.keys()):
        
        z = dic_data[noise_level][effect1]
        z.get_dataloader(n_rolling=T1)

        dic_models, dic_models_rollfalse = get_models()


        lasso_full = ml_benchmarks.CustomBenchmarkRolling(LassoCV(alphas=np.linspace(start=1e-4, stop=1e-2, num=30), 
                                                                verbose=False, max_iter=2000, selection='random', n_jobs=5, cv=3))
        lasso_full.fit(z.train)
        boost_model = ml_benchmarks.CustomBenchmarkRolling(HistGradientBoostingRegressor(max_depth=6, max_iter=500, validation_fraction=0.3, n_iter_no_change=100))
        boost_model.fit(z.train)

        comp = benchmark_comparison.Comparator(models=[lasso_full, boost_model], model_names=['lasso_full', 'boosting'])
        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        # Save results
        records_train.append({
            "noise_level": noise_level,
            "model": 'lasso_full',
            "train_corr_optimal": corr_train.loc['lasso_full', "optimal"]
        })
        records_test.append({
            "noise_level": noise_level,
            "model": 'lasso_full',
            "test_corr_optimal": corr_test.loc['lasso_full', "optimal"]
        })

        records_train.append({
            "noise_level": noise_level,
            "model": 'boosting',
            "train_corr_optimal": corr_train.loc['boosting', "optimal"]
        })
        records_test.append({
            "noise_level": noise_level,
            "model": 'boosting',
            "test_corr_optimal": corr_test.loc['boosting', "optimal"]
        })
        


        for model_key in dic_models.keys():                   
            print(f"Running → {noise_level} | {model_key}")

            z = dic_data[noise_level][effect1]
            if model_key in ["trans_1D_T4", "trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                z.get_dataloader(n_rolling=T1, roll_y=True)
            else:
                z.get_dataloader(n_rolling=T1)

            if model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                lr=0.001/2
            else:
                lr=0.001

            epochs = 20
            if noise_level in ['correl0.01', 'correl0.03']:
                epochs = 40

            model = dic_models[model_key]

            # Model
            if noise_level == 'correl0.01' and model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                z.get_dataloader(n_rolling=T1)
                model = dic_models_rollfalse[model_key]
                epochs = 60
                lr=0.0001

            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer, device=device)

            
            # Train
            wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

            # Compare
            comp = benchmark_comparison.Comparator(models=[wrapper], model_names=["model1"])

            corr_train = comp.correl(z, mode="train", return_values=True)
            corr_test  = comp.correl(z, mode="test",  return_values=True)

            train_corr = corr_train.loc["model1", "optimal"]
            test_corr  = corr_test.loc["model1", "optimal"]

            # Save results
            records_train.append({
                "noise_level": noise_level,
                "model": model_key,
                "train_corr_optimal": train_corr
            })
            records_test.append({
                "noise_level": noise_level,
                "model": model_key,
                "test_corr_optimal": test_corr
            })

        records_train.append({
                    "noise_level": noise_level,
                    "model": 'max_linear_corr',
                    # "train_corr_optimal": run_sim(ElasticNetCV(l1_ratio=1, alphas=np.linspace(1e-6, 1e-2, num=50), cv=3), N=N1*F1*T1, 
                    #                               T=T_max, rho=float(noise_level[6:]), oos=False)
                    'train_corr_optimal': get_ols_corr(N=N1*F1*T1, T=T_max, rho=np.sqrt(0.5*F1*float(noise_level[6:])**2))

                })

        records_test.append({
                    "noise_level": noise_level,
                    "model": 'max_linear_corr',
                    # "test_corr_optimal": run_sim(ElasticNetCV(l1_ratio=1, alphas=np.linspace(1e-6, 1e-2, num=50), cv=3), N=N1*F1*T1, 
                    #                              T=T_max, rho=float(noise_level[6:]), oos=True)
                    'test_corr_optimal': get_ols_corr(N=N1*F1*T1, T=T_max, rho=np.sqrt(0.5*F1*float(noise_level[6:])**2))
                })
    
    # ————————————————————————————————————————
    # 3. Convert to nice DataFrames
    # ————————————————————————————————————————
    df_train = pd.DataFrame(records_train)
    df_test  = pd.DataFrame(records_test)

    # Pivot: rows = noise level, columns = effect type
    train_pivot = df_train.pivot(index="noise_level", columns="model", values="train_corr_optimal")
    test_pivot  = df_test.pivot(index="noise_level", columns="model", values="test_corr_optimal")

    col_order = ["lasso_full", "boosting"] + list(dic_models.keys()) + ['max_linear_corr']
    train_pivot = train_pivot[col_order]
    test_pivot  = test_pivot[col_order]

    return train_pivot, test_pivot

## Running on linear effect

In [20]:
dic_models, dic_models_rollfalse = get_models()

table_train_lin0, table_test_lin0 = run_models('d_lin')

Running → correl0.01 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:04<00:00,  8.06it/s]


Running → correl0.01 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:08<00:00,  4.84it/s]


Running → correl0.01 | trans_1D_T4


...

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:07<00:00,  2.62it/s]


Running → correl0.3 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:33<00:00,  1.66s/it]


Running → correl0.3 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:57<00:00,  2.85s/it]


In [21]:
display(table_train_lin0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(table_test_lin0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

noise_level,correl0.01,correl0.03,correl0.05,correl0.1,correl0.3
model,,,,,
lasso_full,0.022,0.026,0.040,0.130,0.307
boosting,0.017,0.022,0.035,0.075,0.193
MLP_global,0.045,0.099,0.160,0.311,0.698
MLP_2D,0.045,0.098,0.171,0.336,0.742
trans_1D_T4,0.051,0.104,0.196,0.372,0.789
trans_2D_TCTC,0.365,0.124,0.561,0.704,0.985
trans_2D_TCTCTCTC,0.190,0.109,0.383,0.733,0.960
max_linear_corr,0.071,0.208,0.337,0.598,0.989


noise_level,correl0.01,correl0.03,correl0.05,correl0.1,correl0.3
model,,,,,
lasso_full,0.006,0.010,0.033,0.128,0.304
boosting,0.014,-0.004,0.012,0.021,0.184
MLP_global,0.060,0.151,0.216,0.413,0.809
MLP_2D,0.041,0.100,0.203,0.480,0.850
trans_1D_T4,0.079,0.145,0.256,0.459,0.837
trans_2D_TCTC,0.359,0.138,0.564,0.708,0.986
trans_2D_TCTCTCTC,0.188,0.141,0.382,0.742,0.963
max_linear_corr,0.071,0.208,0.337,0.598,0.989


In [22]:
# Saving the data

table_train_lin0.to_csv('table_train_lin.csv', index=True)
table_test_lin0.to_csv('table_test_lin.csv', index=True)

In [23]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on conditional effect

In [24]:
dic_models, dic_models_rollfalse = get_models()

table_train_cond0, table_test_cond0 = run_models('d_cond')

Running → correl0.01 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:05<00:00,  7.92it/s]


Running → correl0.01 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:08<00:00,  4.67it/s]


Running → correl0.01 | trans_1D_T4


...

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:07<00:00,  2.69it/s]


Running → correl0.3 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:28<00:00,  1.40s/it]


Running → correl0.3 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:54<00:00,  2.73s/it]


In [25]:
display(table_train_cond0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(table_test_cond0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

noise_level,correl0.01,correl0.03,correl0.05,correl0.1,correl0.3
model,,,,,
lasso_full,-0.004,0.007,0.012,0.034,0.077
boosting,0.005,0.024,0.039,0.071,0.174
MLP_global,0.034,0.088,0.153,0.297,0.678
MLP_2D,0.034,0.087,0.145,0.283,0.656
trans_1D_T4,0.034,0.088,0.146,0.285,0.650
trans_2D_TCTC,0.064,0.104,0.258,0.533,0.809
trans_2D_TCTCTCTC,0.073,0.095,0.315,0.486,0.783
max_linear_corr,0.071,0.208,0.337,0.598,0.989


noise_level,correl0.01,correl0.03,correl0.05,correl0.1,correl0.3
model,,,,,
lasso_full,0.011,-0.001,0.004,-0.003,0.010
boosting,-0.000,-0.000,0.003,0.005,0.017
MLP_global,0.007,-0.008,0.005,0.010,0.006
MLP_2D,-0.007,0.000,0.011,0.000,0.074
trans_1D_T4,0.005,-0.005,0.009,0.019,0.078
trans_2D_TCTC,0.041,0.080,0.232,0.493,0.756
trans_2D_TCTCTCTC,0.058,0.087,0.310,0.451,0.719
max_linear_corr,0.071,0.208,0.337,0.598,0.989


In [26]:
# Saving the data

#table_train_cond0.to_csv('table_train_cond.csv', index=True)
#table_test_cond0.to_csv('table_test_cond.csv', index=True)

In [27]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on shift effect

In [28]:
dic_models, dic_models_rollfalse = get_models()

table_train_shift0, table_test_shift0 = run_models('d_shift')

Running → correl0.01 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:05<00:00,  7.86it/s]


Running → correl0.01 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:07<00:00,  5.07it/s]


Running → correl0.01 | trans_1D_T4


...

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:07<00:00,  2.57it/s]


Running → correl0.3 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:27<00:00,  1.38s/it]


Running → correl0.3 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:52<00:00,  2.62s/it]


In [29]:
display(table_train_shift0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(table_test_shift0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

noise_level,correl0.01,correl0.03,correl0.05,correl0.1,correl0.3
model,,,,,
lasso_full,0.011,0.012,0.067,0.186,0.307
boosting,0.009,0.014,0.048,0.095,0.194
MLP_global,0.030,0.097,0.172,0.313,0.706
MLP_2D,0.029,0.097,0.178,0.334,0.752
trans_1D_T4,0.033,0.098,0.187,0.346,0.784
trans_2D_TCTC,0.203,0.117,0.427,0.787,0.983
trans_2D_TCTCTCTC,0.157,0.108,0.446,0.771,0.984
max_linear_corr,0.071,0.208,0.337,0.598,0.989


noise_level,correl0.01,correl0.03,correl0.05,correl0.1,correl0.3
model,,,,,
lasso_full,0.007,0.003,0.059,0.165,0.304
boosting,0.017,0.011,-0.002,0.044,0.183
MLP_global,0.036,0.154,0.232,0.414,0.814
MLP_2D,0.010,0.110,0.212,0.464,0.857
trans_1D_T4,0.013,0.040,0.128,0.318,0.810
trans_2D_TCTC,0.195,0.121,0.417,0.787,0.983
trans_2D_TCTCTCTC,0.154,0.110,0.455,0.781,0.985
max_linear_corr,0.071,0.208,0.337,0.598,0.989


In [30]:
# Saving the data

table_train_shift0.to_csv('table_train_shift.csv', index=True)
table_test_shift0.to_csv('table_test_shift.csv', index=True)

In [31]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on the cs effect

In [32]:
dic_models, dic_models_rollfalse = get_models()

table_train_cs0, table_test_cs0 = run_models('d_cs')

Running → correl0.01 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:05<00:00,  7.59it/s]


Running → correl0.01 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:07<00:00,  5.42it/s]


Running → correl0.01 | trans_1D_T4


...

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:07<00:00,  2.54it/s]


Running → correl0.3 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:27<00:00,  1.38s/it]


Running → correl0.3 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:52<00:00,  2.62s/it]


In [33]:
display(table_train_cs0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(table_test_cs0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

noise_level,correl0.01,correl0.03,correl0.05,correl0.1,correl0.3
model,,,,,
lasso_full,0.018,0.027,0.055,0.142,0.299
boosting,0.009,0.026,0.040,0.084,0.189
MLP_global,0.035,0.101,0.151,0.308,0.705
MLP_2D,0.032,0.101,0.163,0.321,0.737
trans_1D_T4,0.037,0.109,0.191,0.375,0.789
trans_2D_TCTC,0.060,0.124,0.397,0.708,0.970
trans_2D_TCTCTCTC,0.083,0.108,0.786,0.778,0.966
max_linear_corr,0.071,0.208,0.337,0.598,0.989


noise_level,correl0.01,correl0.03,correl0.05,correl0.1,correl0.3
model,,,,,
lasso_full,0.015,0.022,0.038,0.135,0.307
boosting,0.009,0.002,0.008,0.043,0.172
MLP_global,0.037,0.130,0.205,0.420,0.819
MLP_2D,0.034,0.108,0.200,0.451,0.845
trans_1D_T4,0.041,0.168,0.242,0.485,0.845
trans_2D_TCTC,0.069,0.144,0.396,0.717,0.970
trans_2D_TCTCTCTC,0.070,0.156,0.786,0.789,0.966
max_linear_corr,0.071,0.208,0.337,0.598,0.989


In [34]:
# Saving the data

table_train_cs0.to_csv('table_train_cs.csv', index=True)
table_test_cs0.to_csv('table_test_cs.csv', index=True)

In [35]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on the cs_shift

In [36]:
dic_models, dic_models_rollfalse = get_models()

table_train_csshift0, table_test_csshift0 = run_models('d_cs_shift')

Running → correl0.01 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:05<00:00,  7.49it/s]


Running → correl0.01 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:07<00:00,  5.31it/s]


Running → correl0.01 | trans_1D_T4


...

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:07<00:00,  2.79it/s]


Running → correl0.3 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:27<00:00,  1.38s/it]


Running → correl0.3 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:52<00:00,  2.64s/it]


In [37]:
display(table_train_csshift0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(table_test_csshift0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

noise_level,correl0.01,correl0.03,correl0.05,correl0.1,correl0.3
model,,,,,
lasso_full,0.015,0.016,0.074,0.177,0.311
boosting,0.009,0.021,0.049,0.092,0.202
MLP_global,0.037,0.096,0.162,0.321,0.704
MLP_2D,0.037,0.095,0.178,0.345,0.745
trans_1D_T4,0.040,0.095,0.178,0.356,0.785
trans_2D_TCTC,0.014,0.080,0.079,0.610,0.970
trans_2D_TCTCTCTC,0.024,0.086,0.019,0.100,0.977
max_linear_corr,0.071,0.208,0.337,0.598,0.989


noise_level,correl0.01,correl0.03,correl0.05,correl0.1,correl0.3
model,,,,,
lasso_full,0.009,0.005,0.059,0.174,0.304
boosting,0.004,0.005,0.027,0.046,0.187
MLP_global,0.043,0.133,0.230,0.437,0.816
MLP_2D,0.007,0.079,0.221,0.476,0.851
trans_1D_T4,0.003,0.029,0.143,0.348,0.811
trans_2D_TCTC,0.003,0.006,0.045,0.581,0.970
trans_2D_TCTCTCTC,-0.009,0.000,0.014,0.051,0.976
max_linear_corr,0.071,0.208,0.337,0.598,0.989


In [38]:
# Saving the data

table_train_csshift0.to_csv('table_train_csshift.csv', index=True)
table_test_csshift0.to_csv('table_test_csshift.csv', index=True)

In [39]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

# Function to create table for all effects

In [40]:
def run_models_all_effect(noise_level1, dic_models, dic_models_rollfalse):

    z = dic_data[noise_level1]["d_all"]
    z.get_dataloader(n_rolling=T1)

    lasso_full = ml_benchmarks.CustomBenchmarkRolling(LassoCV(alphas=np.linspace(start=1e-4, stop=1e-2, num=30), 
                                                            verbose=False, max_iter=2000, selection='random', n_jobs=5, cv=3))
    lasso_full.fit(z.train)
    boost_model = ml_benchmarks.CustomBenchmarkRolling(HistGradientBoostingRegressor(max_depth=6, max_iter=500, validation_fraction=0.3, n_iter_no_change=100))
    boost_model.fit(z.train)

    list_models = [lasso_full, boost_model]


    for model_key in dic_models.keys():                   
        print(f"Running → {noise_level1} | {model_key}")

        if model_key in ["trans_1D_T4", "trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            z.get_dataloader(n_rolling=T1, roll_y=True)
        else:
            z.get_dataloader(n_rolling=T1)

        if model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            lr=0.001/2
        else:
            lr=0.001

        epochs = 20
        if noise_level1 in ['correl0.01', 'correl0.03']:
            epochs = 40

        model = dic_models[model_key]

        # Model
        if noise_level1 == 'correl0.01' and model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            z.get_dataloader(n_rolling=T1)
            model = dic_models_rollfalse[model_key]
            epochs = 60
            lr=0.0001

        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer, device=device)

        
        # Train
        wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

        list_models.append(wrapper)

    comp = benchmark_comparison.Comparator(models=list_models, model_names=["lasso_full", "boosting"] + list(dic_models.keys()))

    corr_train = comp.correl(z, mode="train", return_values=True)
    corr_test  = comp.correl(z, mode="test",  return_values=True)

    return corr_train, corr_test

    

## Run on all noise levels

In [41]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_1, all_effects_test_1 = run_models_all_effect("correl0.01", dic_models, dic_models_rollfalse)

Running → correl0.01 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:04<00:00,  8.18it/s]


Running → correl0.01 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:07<00:00,  5.21it/s]


Running → correl0.01 | trans_1D_T4


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:14<00:00,  2.83it/s]


Running → correl0.01 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [01:21<00:00,  1.36s/it]


Running → correl0.01 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [02:36<00:00,  2.60s/it]


In [42]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_3, all_effects_test_3 = run_models_all_effect("correl0.03", dic_models, dic_models_rollfalse)

Running → correl0.03 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:05<00:00,  7.75it/s]


Running → correl0.03 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:07<00:00,  5.21it/s]


Running → correl0.03 | trans_1D_T4


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:13<00:00,  2.88it/s]


Running → correl0.03 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:54<00:00,  1.37s/it]


Running → correl0.03 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [01:45<00:00,  2.64s/it]


In [43]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_5, all_effects_test_5 = run_models_all_effect("correl0.05", dic_models, dic_models_rollfalse)

Running → correl0.05 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:02<00:00,  8.05it/s]


Running → correl0.05 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:04<00:00,  4.93it/s]


Running → correl0.05 | trans_1D_T4


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:07<00:00,  2.79it/s]


Running → correl0.05 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:27<00:00,  1.40s/it]


Running → correl0.05 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:52<00:00,  2.62s/it]


In [44]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_10, all_effects_test_10 = run_models_all_effect("correl0.1", dic_models, dic_models_rollfalse)

Running → correl0.1 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:02<00:00,  7.79it/s]


Running → correl0.1 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:03<00:00,  5.05it/s]


Running → correl0.1 | trans_1D_T4


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:07<00:00,  2.73it/s]


Running → correl0.1 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:27<00:00,  1.39s/it]


Running → correl0.1 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:52<00:00,  2.61s/it]


In [45]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_30, all_effects_test_30 = run_models_all_effect("correl0.3", dic_models, dic_models_rollfalse)

Running → correl0.3 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:02<00:00,  8.16it/s]


Running → correl0.3 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:04<00:00,  4.90it/s]


Running → correl0.3 | trans_1D_T4


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:07<00:00,  2.71it/s]


Running → correl0.3 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:27<00:00,  1.38s/it]


Running → correl0.3 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:52<00:00,  2.60s/it]


In [46]:
display(all_effects_train_1.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.042,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.016,0.450,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.022,0.453,-0.002,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.027,0.439,-0.001,0.004,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.016,0.447,0.005,0.008,-0.012,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.012,0.450,0.009,0.004,-0.008,-0.001,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.108,0.012,0.006,0.005,0.002,0.006,0.007,nan,nan,nan,nan,nan,nan
boosting,0.255,0.022,0.011,0.005,0.006,0.008,0.018,0.292,nan,nan,nan,nan,nan
MLP_global,0.989,0.041,0.015,0.022,0.027,0.018,0.012,0.106,0.250,nan,nan,nan,nan
MLP_2D,0.977,0.041,0.016,0.022,0.027,0.015,0.011,0.104,0.246,0.966,nan,nan,nan


In [47]:
display(all_effects_test_1.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.048,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.023,0.450,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.031,0.451,0.004,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.005,0.452,0.006,0.001,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.025,0.449,0.000,0.003,0.008,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.024,0.446,-0.001,0.004,0.004,-0.003,nan,nan,nan,nan,nan,nan,nan
lasso_full,-0.009,-0.009,0.003,-0.014,-0.018,0.008,0.002,nan,nan,nan,nan,nan,nan
boosting,-0.002,0.004,-0.013,0.000,0.006,0.010,0.005,0.154,nan,nan,nan,nan,nan
MLP_global,0.004,0.047,0.023,0.012,0.023,0.038,0.009,0.067,0.031,nan,nan,nan,nan
MLP_2D,-0.005,0.030,0.011,0.002,0.021,0.011,0.023,0.029,0.016,0.310,nan,nan,nan


In [48]:
display(all_effects_train_3.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.093,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.034,0.448,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.041,0.450,0.007,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.047,0.453,0.005,0.001,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.050,0.450,0.001,0.002,0.004,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.037,0.442,-0.005,-0.002,0.005,-0.003,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.108,0.029,0.028,-0.001,0.007,0.023,0.009,nan,nan,nan,nan,nan,nan
boosting,0.252,0.026,0.015,0.005,0.015,0.016,0.008,0.288,nan,nan,nan,nan,nan
MLP_global,0.989,0.096,0.035,0.041,0.049,0.051,0.040,0.108,0.249,nan,nan,nan,nan
MLP_2D,0.977,0.091,0.032,0.040,0.046,0.050,0.036,0.110,0.248,0.967,nan,nan,nan


In [49]:
display(all_effects_test_3.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.088,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.038,0.451,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.042,0.448,0.007,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.034,0.436,0.007,-0.015,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.040,0.441,-0.001,-0.009,-0.008,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.042,0.453,-0.005,0.019,-0.009,-0.001,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.013,0.026,0.032,-0.012,0.005,0.028,0.003,nan,nan,nan,nan,nan,nan
boosting,0.002,0.014,-0.007,0.001,0.014,0.026,-0.004,0.128,nan,nan,nan,nan,nan
MLP_global,0.016,0.118,0.057,-0.003,0.065,0.084,0.060,0.105,0.039,nan,nan,nan,nan
MLP_2D,-0.002,0.048,0.017,-0.010,0.043,0.040,0.017,0.054,0.023,0.326,nan,nan,nan


In [50]:
display(all_effects_train_5.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.157,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.071,0.452,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.082,0.452,0.005,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.070,0.457,0.009,0.018,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.068,0.448,-0.005,0.005,0.008,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.063,0.452,0.008,0.001,0.005,0.003,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.106,0.032,0.038,0.006,0.018,-0.005,0.015,nan,nan,nan,nan,nan,nan
boosting,0.257,0.033,0.019,0.020,0.017,0.010,0.010,0.295,nan,nan,nan,nan,nan
MLP_global,0.984,0.161,0.073,0.080,0.072,0.073,0.066,0.108,0.252,nan,nan,nan,nan
MLP_2D,0.940,0.162,0.073,0.080,0.074,0.072,0.068,0.105,0.238,0.927,nan,nan,nan


In [51]:
display(all_effects_test_5.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.161,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.070,0.451,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.078,0.460,0.008,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.072,0.463,0.003,0.026,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.066,0.447,0.006,0.004,0.011,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.080,0.449,0.012,0.003,0.007,-0.004,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.017,0.029,0.035,0.005,0.008,0.011,0.007,nan,nan,nan,nan,nan,nan
boosting,0.015,0.021,0.021,-0.001,0.005,0.009,0.013,0.172,nan,nan,nan,nan,nan
MLP_global,0.025,0.167,0.101,0.014,0.080,0.099,0.086,0.083,0.041,nan,nan,nan,nan
MLP_2D,0.024,0.115,0.069,0.001,0.074,0.059,0.060,0.047,0.021,0.347,nan,nan,nan


In [52]:
display(all_effects_train_10.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.300,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.148,0.470,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.138,0.462,0.023,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.135,0.469,0.024,0.023,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.130,0.460,0.025,0.016,0.013,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.145,0.461,0.020,0.006,0.026,0.017,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.126,0.145,0.108,0.022,0.073,0.088,0.046,nan,nan,nan,nan,nan,nan
boosting,0.257,0.085,0.042,0.040,0.039,0.042,0.035,0.339,nan,nan,nan,nan,nan
MLP_global,0.983,0.310,0.153,0.135,0.141,0.136,0.153,0.129,0.249,nan,nan,nan,nan
MLP_2D,0.940,0.329,0.168,0.132,0.154,0.148,0.161,0.131,0.243,0.928,nan,nan,nan


In [53]:
display(all_effects_test_10.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.299,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.152,0.466,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.146,0.461,0.009,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.139,0.463,0.022,0.017,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.120,0.460,0.016,0.013,0.013,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.136,0.470,0.034,0.016,0.030,0.021,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.019,0.099,0.092,-0.011,0.067,0.065,0.018,nan,nan,nan,nan,nan,nan
boosting,0.011,0.024,0.011,0.010,0.017,0.011,0.007,0.193,nan,nan,nan,nan,nan
MLP_global,0.112,0.349,0.209,0.021,0.191,0.187,0.204,0.117,0.048,nan,nan,nan,nan
MLP_2D,0.098,0.353,0.212,0.020,0.192,0.191,0.208,0.077,0.031,0.404,nan,nan,nan


In [54]:
display(all_effects_train_30.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.691,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.391,0.568,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.392,0.567,0.155,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.391,0.567,0.154,0.149,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.392,0.566,0.145,0.153,0.151,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.395,0.570,0.153,0.151,0.154,0.159,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.212,0.279,0.184,0.074,0.160,0.186,0.186,nan,nan,nan,nan,nan,nan
boosting,0.258,0.185,0.109,0.100,0.097,0.107,0.111,0.596,nan,nan,nan,nan,nan
MLP_global,0.987,0.703,0.400,0.385,0.402,0.402,0.404,0.223,0.261,nan,nan,nan,nan
MLP_2D,0.956,0.733,0.430,0.378,0.428,0.419,0.423,0.229,0.250,0.949,nan,nan,nan


In [55]:
display(all_effects_test_30.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.687,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.374,0.553,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.393,0.568,0.138,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.390,0.565,0.148,0.148,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.405,0.568,0.135,0.170,0.143,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.377,0.565,0.141,0.151,0.150,0.152,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.193,0.288,0.181,0.089,0.152,0.192,0.198,nan,nan,nan,nan,nan,nan
boosting,0.113,0.156,0.106,0.045,0.075,0.096,0.119,0.556,nan,nan,nan,nan,nan
MLP_global,0.504,0.736,0.476,0.202,0.473,0.465,0.459,0.256,0.145,nan,nan,nan,nan
MLP_2D,0.531,0.783,0.497,0.232,0.511,0.477,0.490,0.267,0.154,0.732,nan,nan,nan


In [56]:
# Let's also save all the tables above.
all_effects_train_1.to_csv('all_effects_train_1.csv', index=True)
all_effects_test_1.to_csv('all_effects_test_1.csv', index=True)

all_effects_train_3.to_csv('all_effects_train_3.csv', index=True)
all_effects_test_3.to_csv('all_effects_test_3.csv', index=True)

all_effects_train_5.to_csv('all_effects_train_5.csv', index=True)
all_effects_test_5.to_csv('all_effects_test_5.csv', index=True)

all_effects_train_10.to_csv('all_effects_train_10.csv', index=True)
all_effects_test_10.to_csv('all_effects_test_10.csv', index=True)

all_effects_train_30.to_csv('all_effects_train_30.csv', index=True)
all_effects_test_30.to_csv('all_effects_test_30.csv', index=True)

# Testing sparsity

In [57]:
# To test sparsity let's write a function that runs one model on all the effects and noise levels.
# Then we can run on the model with and without sparsity.

In [58]:
def keep_topk_per_row(x, k=3):
    vals, idx = torch.topk(x, k=k, dim=-1, largest=True)
    out = torch.zeros_like(x)
    out.scatter_(-1, idx, 1)
    return out

In [59]:
def test_sparsity_all_correl_effets(sparsity=False):

    # Storage
    records_train = []
    records_test  = []

    for noise_level in dic_data.keys():          
        for effect in ['d_lin', 'd_cond', 'd_shift', 'd_cs', 'd_cs_shift', 'd_all']:                   
            print(f"Running → {noise_level} | {effect}")

            z = dic_data[noise_level][effect]
            z.get_dataloader(n_rolling=T1, roll_y=True)
            
            lr=0.001/2
            roll_y=True
            
            epochs = 20
            if noise_level in ['correl0.01', 'correl0.03']:
                epochs = 40


            # Model
            if noise_level == 'correl0.01':
                z.get_dataloader(n_rolling=T1)
                roll_y=False
                epochs = 60
                lr=0.0001

            # Model
            if sparsity:
                model = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                    dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=keep_topk_per_row, 
                                                                               roll_y=roll_y).to(device)
            else:
                model = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                                roll_y=roll_y).to(device)
            
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer, device=device)

            # Train
            wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

            # Compare
            comp = benchmark_comparison.Comparator(models=[wrapper], model_names=["model1"])

            corr_train = comp.correl(z, mode="train", return_values=True)
            corr_test  = comp.correl(z, mode="test",  return_values=True)

            train_corr = corr_train.loc["model1", "optimal"]
            test_corr  = corr_test.loc["model1", "optimal"]

            # Save results
            records_train.append({
                "noise_level": noise_level,
                "effect": effect,
                "train_corr_optimal": train_corr
            })
            records_test.append({
                "noise_level": noise_level,
                "effect": effect,
                "test_corr_optimal": test_corr
            })

    # ————————————————————————————————————————
    # 3. Convert to nice DataFrames
    # ————————————————————————————————————————
    df_train = pd.DataFrame(records_train)
    df_test  = pd.DataFrame(records_test)

    # Pivot: rows = noise level, columns = effect type
    train_pivot = df_train.pivot(index="noise_level", columns="effect", values="train_corr_optimal")
    test_pivot  = df_test.pivot(index="noise_level", columns="effect", values="test_corr_optimal")    

    # Sort columns in logical order
    col_order = ["d_lin", "d_cond", "d_shift", "d_cs", "d_cs_shift", "d_all"]
    train_model = train_pivot[col_order]
    test_model  = test_pivot[col_order]

    return train_model, test_model

In [60]:
table_train_no_sparsity, table_test_no_sparsity = test_sparsity_all_correl_effets()

Running → correl0.01 | d_lin


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [01:21<00:00,  1.36s/it]


Running → correl0.01 | d_cond


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [01:21<00:00,  1.36s/it]


Running → correl0.01 | d_shift


...

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:27<00:00,  1.39s/it]


Running → correl0.3 | d_cs_shift


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:27<00:00,  1.39s/it]


Running → correl0.3 | d_all


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:27<00:00,  1.38s/it]


In [61]:
table_train_with_sparsity, table_test_with_sparsity = test_sparsity_all_correl_effets(sparsity=True)

Running → correl0.01 | d_lin


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [01:22<00:00,  1.37s/it]


Running → correl0.01 | d_cond


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [01:21<00:00,  1.36s/it]


Running → correl0.01 | d_shift


...

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:27<00:00,  1.38s/it]


Running → correl0.3 | d_cs_shift


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:27<00:00,  1.40s/it]


Running → correl0.3 | d_all


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:27<00:00,  1.38s/it]


In [62]:
display(table_train_no_sparsity.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

effect,d_lin,d_cond,d_shift,d_cs,d_cs_shift,d_all
noise_level,,,,,,
correl0.01,0.273,0.075,0.259,0.054,0.023,0.141
correl0.03,0.110,0.099,0.118,0.130,0.089,0.114
correl0.05,0.444,0.274,0.377,0.428,0.026,0.320
correl0.1,0.728,0.570,0.670,0.698,0.695,0.703
correl0.3,0.974,0.803,0.982,0.973,0.971,0.938


In [63]:
display(table_train_with_sparsity.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

effect,d_lin,d_cond,d_shift,d_cs,d_cs_shift,d_all
noise_level,,,,,,
correl0.01,0.243,0.052,0.204,0.078,0.011,0.170
correl0.03,0.127,0.097,0.125,0.118,0.083,0.104
correl0.05,0.430,0.302,0.411,0.450,0.047,0.338
correl0.1,0.774,0.520,0.665,0.693,0.627,0.607
correl0.3,0.973,0.814,0.982,0.971,0.976,0.940


In [64]:
display(table_test_no_sparsity.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

effect,d_lin,d_cond,d_shift,d_cs,d_cs_shift,d_all
noise_level,,,,,,
correl0.01,0.268,0.066,0.259,0.059,0.005,0.125
correl0.03,0.129,0.089,0.136,0.141,0.008,0.114
correl0.05,0.436,0.261,0.372,0.425,0.014,0.295
correl0.1,0.730,0.543,0.675,0.704,0.683,0.701
correl0.3,0.974,0.740,0.982,0.973,0.970,0.935


In [65]:
display(table_test_with_sparsity.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

effect,d_lin,d_cond,d_shift,d_cs,d_cs_shift,d_all
noise_level,,,,,,
correl0.01,0.252,0.037,0.198,0.068,0.007,0.176
correl0.03,0.153,0.088,0.121,0.141,-0.002,0.103
correl0.05,0.437,0.281,0.411,0.463,0.030,0.316
correl0.1,0.776,0.495,0.666,0.706,0.600,0.588
correl0.3,0.973,0.757,0.982,0.970,0.975,0.937


In [66]:
# To store the tables above

table_train_no_sparsity.to_csv('table_train_no_sparsity.csv', index=True)
table_test_no_sparsity.to_csv('table_test_no_sparsity.csv', index=True)

table_train_with_sparsity.to_csv('table_train_with_sparsity.csv', index=True)
table_test_with_sparsity.to_csv('table_test_with_sparsity.csv', index=True)

### Sparsity boostrap

In [67]:
dic_data

{'correl0.01': {'d_lin': <tsnn.generators.generators.Generator at 0x175ceb430>,
  'd_cond': <tsnn.generators.generators.Generator at 0x175ceadd0>,
  'd_shift': <tsnn.generators.generators.Generator at 0x175d11180>,
  'd_cs': <tsnn.generators.generators.Generator at 0x175d11270>,
  'd_cs_shift': <tsnn.generators.generators.Generator at 0x175d11150>,
  'd_all': <tsnn.generators.generators.Generator at 0x175d11120>},
 'correl0.03': {'d_lin': <tsnn.generators.generators.Generator at 0x175d110f0>,
  'd_cond': <tsnn.generators.generators.Generator at 0x175d110c0>,
  'd_shift': <tsnn.generators.generators.Generator at 0x175d11090>,
  'd_cs': <tsnn.generators.generators.Generator at 0x175d11060>,
  'd_cs_shift': <tsnn.generators.generators.Generator at 0x175d11030>,
  'd_all': <tsnn.generators.generators.Generator at 0x175d10190>},
 'correl0.05': {'d_lin': <tsnn.generators.generators.Generator at 0x175d10fd0>,
  'd_cond': <tsnn.generators.generators.Generator at 0x175d11000>,
  'd_shift': <tsn

In [68]:
def bootstrap_sparsity(noise_level='correl0.03', effect='d_all', n=10):

    # Storage
    records_train = []
    records_test  = []               
    print(f"Running → {noise_level} | {effect}")

    z = dic_data[noise_level][effect]
    z.get_dataloader(n_rolling=T1, roll_y=True)

    lr=0.001/2
    roll_y=True

    epochs = 20
    if noise_level in ['correl0.01', 'correl0.03']:
        epochs = 40

    for k in tqdm(range(n)):
        # Model
        if noise_level == 'correl0.01':
            z.get_dataloader(n_rolling=T1)
            roll_y=False
            epochs = 60
            lr=0.0001

        model_sparse = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=keep_topk_per_row, 
                                                                           roll_y=roll_y).to(device)

        model = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                            roll_y=roll_y).to(device)

        ##################### not sparse
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer, device=device)

        # Train
        wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False, verbose=0)

        # Compare
        comp = benchmark_comparison.Comparator(models=[wrapper], model_names=["model1"])

        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        train_corr = corr_train.loc["model1", "optimal"]
        test_corr  = corr_test.loc["model1", "optimal"]
        
        ###################### sparse
        optimizer = torch.optim.Adam(model_sparse.parameters(), lr=lr)
        wrapper = torch_benchmarks.TorchWrapper(model_sparse, optimizer=optimizer, device=device)

        # Train
        wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False, verbose=0)

        # Compare
        comp = benchmark_comparison.Comparator(models=[wrapper], model_names=["model1"])

        corr_train_sparse = comp.correl(z, mode="train", return_values=True)
        corr_test_sparse  = comp.correl(z, mode="test",  return_values=True)

        train_corr_sparse = corr_train_sparse.loc["model1", "optimal"]
        test_corr_sparse  = corr_test_sparse.loc["model1", "optimal"]

        # Save results
        records_train.append({
            "train_corr_sparse": train_corr_sparse,
            "train_corr": train_corr
        })
        records_test.append({
            "test_corr_sparse": test_corr_sparse,
            "test_corr": test_corr
        })

    # ————————————————————————————————————————
    # 3. Convert to nice DataFrames
    # ————————————————————————————————————————
    df_train = pd.DataFrame(records_train)
    df_test  = pd.DataFrame(records_test)

    return df_train, df_test

#### Correl 0.01

In [ ]:
for effect in dic_data['correl0.01']:
    table_train, table_test = bootstrap_sparsity(noise_level='correl0.01', effect=effect, n=40)
    display(pd.concat([table_test.quantile([0.1, 0.9]), table_test.aggregate(['mean', 'std'])]).T.rename_axis(effect, axis=0).style.background_gradient().format(precision=4))

Running → correl0.01 | d_lin


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [1:53:02<00:00, 169.56s/it]


,0.100000,0.900000,mean,std
d_lin,,,,
test_corr_sparse,0.2005,0.3159,0.2476,0.0449
test_corr,0.2087,0.3378,0.2612,0.0591


Running → correl0.01 | d_cond


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [1:52:56<00:00, 169.41s/it]


...

Running → correl0.01 | d_cs


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [1:54:22<00:00, 171.56s/it]


,0.100000,0.900000,mean,std
d_cs,,,,
test_corr_sparse,0.0448,0.0780,0.0601,0.0135
test_corr,0.0438,0.0783,0.0610,0.0134


Running → correl0.01 | d_cs_shift


 12%|████████████▋                                                                                        | 5/40 [14:10<1:39:32, 170.65s/it]

#### Correl 0.03

In [ ]:
for effect in dic_data['correl0.03']:
    table_train, table_test = bootstrap_sparsity(noise_level='correl0.03', effect=effect, n=40)
    display(pd.concat([table_test.quantile([0.1, 0.9]), table_test.aggregate(['mean', 'std'])]).T.rename_axis(effect, axis=0).style.background_gradient().format(precision=4))

#### Correl 0.05

In [ ]:
for effect in dic_data['correl0.05']:
    table_train, table_test = bootstrap_sparsity(noise_level='correl0.05', effect=effect, n=40)
    display(pd.concat([table_test.quantile([0.1, 0.9]), table_test.aggregate(['mean', 'std'])]).T.rename_axis(effect, axis=0).style.background_gradient().format(precision=4))

#### Correl 0.1

In [ ]:
for effect in dic_data['correl0.1']:
    table_train, table_test = bootstrap_sparsity(noise_level='correl0.1', effect=effect, n=40)
    display(pd.concat([table_test.quantile([0.1, 0.9]), table_test.aggregate(['mean', 'std'])]).T.rename_axis(effect, axis=0).style.background_gradient().format(precision=4))

#### Correl 0.3

In [ ]:
for effect in dic_data['correl0.3']:
    table_train, table_test = bootstrap_sparsity(noise_level='correl0.3', effect=effect, n=40)
    display(pd.concat([table_test.quantile([0.1, 0.9]), table_test.aggregate(['mean', 'std'])]).T.rename_axis(effect, axis=0).style.background_gradient().format(precision=4))